## Teste da tese ViaMar: saturacao estrutural x sazonalidade turistica

**Projeto:** Diagnostico de acidentes para o estudo de impacto da ViaMar (Joinville-Florianopolis)

**Autor:** Lucas Handerson

**Fonte dos dados:** Acidentes PRF (datatran2024/2025/2026, acidentes2026); volume de trafego
DNIT PNCT (vmda2022_snv_202301b, contagem_trafego_2020); frota SENATRAN (frota-dezembro2020,
FrotaporMunicipioetipoDEZEMBRO2025); populacao residente IBGE Censo 2022 (populacao_corredor);
qualidade do pavimento CNT 2025 (cnt_pesquisa_rodovias_2025).

### Contexto

Estende o diagnostico de acidentes de analise_preliminar/analise_preliminar.ipynb, que
mostrou que o corredor Garuva-Palhoca concentra ~83% dos acidentes da BR-101/SC de forma estavel
ano a ano. Aqui testamos se essa saturacao e estrutural (trafego, frota, pendularidade) ou
apenas sazonal (turismo de verao), cruzando os acidentes PRF (2024-2026) com:

  1. Volume de trafego (VMD/VMDa, DNIT PNCT 2020/2022) por trecho de km
  2. Frota de veiculos por municipio (SENATRAN, Dez/2020 vs Dez/2025)
  3. Perfil horario de pico + populacao residente 2022 (proxy de pendularidade)
  4. Tipo de veiculo envolvido (caminhao x passeio), proxy de trafego de carga
  5. Qualidade do pavimento (CNT 2025, nivel estadual)

Cada secao carrega uma fonte de dados, calcula os numeros usados no resumo executivo e gera a
figura correspondente. Roda de forma independente (sem notebook): le tudo de ../dados e escreve
as figuras em ../output/figs.

### Setup

In [ ]:
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

ROOT = Path.cwd().parent
DADOS_DIR = ROOT / "dados"
FIGS_DIR = ROOT / "output" / "figs"
FIGS_DIR.mkdir(parents=True, exist_ok=True)

COR_DESTAQUE = "#c0392b"
COR_BASE = "#2a78d6"
COR_NEUTRA = "#7f8c8d"

CORREDOR_MUNICIPIOS = [
    "GARUVA", "JOINVILLE", "ARAQUARI", "BARRA VELHA", "BALNEARIO PICARRAS",
    "PENHA", "NAVEGANTES", "ITAJAI", "CAMBORIU", "BALNEARIO CAMBORIU",
    "ITAPEMA", "PORTO BELO", "TIJUCAS", "GOVERNADOR CELSO RAMOS", "BIGUACU",
    "SAO JOSE", "FLORIANOPOLIS", "PALHOCA",
]
MUNICIPIO_IBGE = {
    "ARAQUARI": 4201307, "BALNEARIO CAMBORIU": 4202008, "BARRA VELHA": 4202107,
    "BIGUACU": 4202305, "CAMBORIU": 4203204, "FLORIANOPOLIS": 4205407,
    "GARUVA": 4205803, "GOVERNADOR CELSO RAMOS": 4206009, "ITAJAI": 4208203,
    "ITAPEMA": 4208302, "JOINVILLE": 4209102, "NAVEGANTES": 4211306,
    "PALHOCA": 4211900, "PENHA": 4212502, "BALNEARIO PICARRAS": 4212809,
    "PORTO BELO": 4213500, "SAO JOSE": 4216602, "TIJUCAS": 4218004,
}
CORREDOR_KM_MIN, CORREDOR_KM_MAX = 0, 232


def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn").upper().strip()

### 0 - Acidentes PRF 2024-2026, recorte do corredor Joinville-Palhoca

In [ ]:
def carregar_datatran(nome_arquivo, formato_data, ano):
    df = pd.read_csv(DADOS_DIR / nome_arquivo, sep=";", encoding="latin1", low_memory=False)
    df["data_inversa"] = pd.to_datetime(df["data_inversa"], format=formato_data, errors="coerce")
    df["km_num"] = pd.to_numeric(df["km"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    df["ano"] = ano
    return df


print("=" * 70)
print("0 - CARREGANDO ACIDENTES (PRF, 2024-2026)")
print("=" * 70)

dt = pd.concat([
    carregar_datatran("datatran2024.csv", "%Y-%m-%d", 2024),
    carregar_datatran("datatran2025.csv", "%Y-%m-%d", 2025),
    carregar_datatran("datatran2026.csv", "%d/%m/%Y", 2026),
], ignore_index=True)

br101_sc = dt[(dt["br"] == 101) & (dt["uf"] == "SC")].copy()
corredor = br101_sc[br101_sc["municipio"].isin(CORREDOR_MUNICIPIOS)].copy()
print(f"Acidentes no corredor (3 anos, 2026 parcial): {len(corredor):,}")

ac2026 = pd.read_csv(DADOS_DIR / "acidentes2026.csv", sep=";", encoding="latin1", low_memory=False)
corredor_ac2026 = ac2026[ac2026["id"].isin(corredor.loc[corredor["ano"] == 2026, "id"])]

### 1 - Volume de trafego (DNIT) x densidade de acidentes por trecho de km

In [ ]:
print("\n" + "=" * 70)
print("1 - VOLUME DE TRAFEGO (DNIT VMDa) x DENSIDADE DE ACIDENTES POR TRECHO")
print("=" * 70)


def carregar_vmd_com_km(caminho, sheet_name=None):
    if sheet_name:
        df = pd.read_excel(caminho, sheet_name=sheet_name)
    else:
        df = pd.read_csv(caminho, sep=None, engine="python")
    sub = df[(df["vl_br"] == 101) & (df["sg_uf"] == "SC")].copy()

    def to_num(col):
        return pd.to_numeric(sub[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")

    sub["vl_km_inic"] = to_num("vl_km_inic")
    sub["vl_km_fina"] = to_num("vl_km_fina")
    sub = sub.dropna(subset=["vl_km_inic", "vl_km_fina"])
    sub = sub[sub["vl_km_inic"] < CORREDOR_KM_MAX]
    for c in ("VMDa_C", "VMDa_D"):
        sub[c] = to_num(c)
    sub["vmda_total"] = sub["VMDa_C"].fillna(0) + sub["VMDa_D"].fillna(0)
    # varios registros podem cobrir o mesmo trecho (ex.: "Referencial" duplicado) -> agrega por km
    agg = (sub[sub["vmda_total"] > 0]
           .groupby(["vl_km_inic", "vl_km_fina"], as_index=False)
           .agg(vmda_total=("vmda_total", "mean"), ds_local_i=("ds_local_i", "first")))
    return agg.sort_values("vl_km_inic")


vmd_2022 = carregar_vmd_com_km(DADOS_DIR / "vmda2022_snv_202301b.xlsx", sheet_name="SNV202301B")
vmd_2020 = carregar_vmd_com_km(DADOS_DIR / "contagem_trafego_2020.csv")
print(f"Trechos com VMD válido no corredor -- 2022: {len(vmd_2022)}, 2020: {len(vmd_2020)}")

# acidentes por trecho: para cada segmento VMD, conta acidentes do corredor cujo km cai no intervalo
acid_por_km = corredor["km_num"].dropna()


def acidentes_no_trecho(km_i, km_f, serie_km):
    return int(((serie_km >= km_i) & (serie_km < km_f)).sum())


vmd_2022["extensao"] = (vmd_2022["vl_km_fina"] - vmd_2022["vl_km_inic"]).clip(lower=0.1)
vmd_2022["acidentes_3anos"] = [
    acidentes_no_trecho(i, f, acid_por_km) for i, f in zip(vmd_2022["vl_km_inic"], vmd_2022["vl_km_fina"])
]
vmd_2022["acidentes_por_km_ano"] = vmd_2022["acidentes_3anos"] / vmd_2022["extensao"] / 3

corr_vmd_acid = vmd_2022[["vmda_total", "acidentes_por_km_ano"]].corr().iloc[0, 1]
print(f"Correlação VMDa x densidade de acidentes (por trecho, corredor): r = {corr_vmd_acid:.2f}")

vmd_2022_media = vmd_2022["vmda_total"].mean()
vmd_2020_media = vmd_2020["vmda_total"].mean()
crescimento_vmd = (vmd_2022_media / vmd_2020_media - 1) * 100 if vmd_2020_media else float("nan")
print(f"VMDa médio corredor -- 2020: {vmd_2020_media:,.0f} veíc/dia | 2022: {vmd_2022_media:,.0f} veíc/dia "
      f"({crescimento_vmd:+.1f}% em 2 anos)")

# nivel de servico (LOS) so vem no arquivo de 2020 -- D/E/F = fluxo proximo ou alem da capacidade
ns_2020 = pd.read_csv(DADOS_DIR / "contagem_trafego_2020.csv", sep=None, engine="python")
ns_2020 = ns_2020[(ns_2020["vl_br"] == 101) & (ns_2020["sg_uf"] == "SC")].copy()
ns_2020["vl_km_inic"] = pd.to_numeric(ns_2020["vl_km_inic"].astype(str).str.replace(",", "."), errors="coerce")
ns_2020["vl_km_fina"] = pd.to_numeric(ns_2020["vl_km_fina"].astype(str).str.replace(",", "."), errors="coerce")
ns_2020 = ns_2020.dropna(subset=["vl_km_inic", "NS_C"])
ns_2020 = ns_2020[ns_2020["vl_km_inic"] < CORREDOR_KM_MAX]
trechos_d_ou_pior = ns_2020[ns_2020["NS_C"].isin(["D", "E", "F"]) | ns_2020["NS_D"].isin(["D", "E", "F"])]
print(f"Trechos com Nível de Serviço D (próximo da capacidade) em Dez/2020: {len(trechos_d_ou_pior)} de {len(ns_2020)}")
for _, r in trechos_d_ou_pior.sort_values("vl_km_inic").iterrows():
    print(f"  km {r['vl_km_inic']:.1f}-{r['vl_km_fina']:.1f} ({r['ds_local_i']}): NS_C={r['NS_C']}, NS_D={r['NS_D']}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.bar(vmd_2022["vl_km_inic"], vmd_2022["vmda_total"], width=(vmd_2022["extensao"]) * 0.9,
       align="edge", color=COR_BASE, alpha=0.75, label="VMDa 2022 (veíc/dia)")
ax2 = ax.twinx()
ax2.plot(vmd_2022["vl_km_inic"] + vmd_2022["extensao"] / 2, vmd_2022["acidentes_por_km_ano"],
          color=COR_DESTAQUE, marker="o", lw=2, label="Densidade de acidentes")
ax.set_xlabel("km (BR-101/SC, Garuvá→Palhoça)")
ax.set_ylabel("VMDa 2022 (veíc/dia)", color=COR_BASE)
ax2.set_ylabel("Acidentes / km / ano", color=COR_DESTAQUE)
ax.set_title("Volume de tráfego e densidade de acidentes seguem o mesmo perfil espacial")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

ax = axes[1]
ax.scatter(vmd_2022["vmda_total"], vmd_2022["acidentes_por_km_ano"], color=COR_BASE, s=50, alpha=0.8)
if len(vmd_2022) > 2:
    z = np.polyfit(vmd_2022["vmda_total"], vmd_2022["acidentes_por_km_ano"], 1)
    xs = np.linspace(vmd_2022["vmda_total"].min(), vmd_2022["vmda_total"].max(), 50)
    ax.plot(xs, np.polyval(z, xs), color=COR_DESTAQUE, ls="--", label=f"r = {corr_vmd_acid:.2f}")
ax.set_xlabel("VMDa 2022 (veíc/dia)")
ax.set_ylabel("Acidentes / km / ano")
ax.set_title("Correlação volume x densidade de acidentes por trecho")
ax.legend()

plt.tight_layout()
plt.savefig(FIGS_DIR / "fig1_vmd_x_acidentes.png", dpi=150)
plt.close()

### 2 - Frota de veiculos (SENATRAN 2020 vs 2025) x crescimento de acidentes

In [ ]:
print("\n" + "=" * 70)
print("2 - CRESCIMENTO DA FROTA (SENATRAN, 2020 -> 2025) x ACIDENTES NO CORREDOR")
print("=" * 70)


def carregar_frota(caminho, sheet_name):
    df = pd.read_excel(caminho, sheet_name=sheet_name, skiprows=2)
    df.columns = [str(c).strip().upper() for c in df.columns]
    df["UF"] = df["UF"].astype(str).str.strip()
    df["MUNICIPIO_NORM"] = df["MUNICIPIO"].apply(strip_accents)
    return df[(df["UF"] == "SC") & (df["MUNICIPIO_NORM"].isin(CORREDOR_MUNICIPIOS))]


frota_2020 = carregar_frota(DADOS_DIR / "frota-dezembro2020.xls", "DEZEMBRO_2020")
frota_2025 = carregar_frota(DADOS_DIR / "FrotaporMunicipioetipoDEZEMBRO2025.xlsx", "DEZEMBRO_2025")

frota_2020_total = frota_2020["TOTAL"].sum()
frota_2025_total = frota_2025["TOTAL"].sum()
crescimento_frota = (frota_2025_total / frota_2020_total - 1) * 100
print(f"Frota corredor -- Dez/2020: {frota_2020_total:,.0f} | Dez/2025: {frota_2025_total:,.0f} "
      f"({crescimento_frota:+.1f}% em 5 anos)")

acid_2024 = int((corredor["ano"] == 2024).sum())
acid_2025 = int((corredor["ano"] == 2025).sum())
crescimento_acid_1ano = (acid_2025 / acid_2024 - 1) * 100
print(f"Acidentes corredor -- 2024: {acid_2024:,} | 2025: {acid_2025:,} ({crescimento_acid_1ano:+.1f}% em 1 ano)")
print("(frota é comparação 2020->2025 / 5 anos; acidentes só têm série 2024-2026 -- janelas não coincidem, "
      "ver ressalva no resumo executivo)")

frota_por_mun = (frota_2025.set_index("MUNICIPIO_NORM")["TOTAL"]
                  .rename("2025")
                  .to_frame()
                  .join(frota_2020.set_index("MUNICIPIO_NORM")["TOTAL"].rename("2020")))
frota_por_mun["crescimento_pct"] = (frota_por_mun["2025"] / frota_por_mun["2020"] - 1) * 100
frota_por_mun = frota_por_mun.sort_values("2025", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

ax = axes[0]
y = np.arange(len(frota_por_mun))
ax.barh(y - 0.2, frota_por_mun["2020"], height=0.4, color=COR_NEUTRA, label="Dez/2020")
ax.barh(y + 0.2, frota_por_mun["2025"], height=0.4, color=COR_BASE, label="Dez/2025")
ax.set_yticks(y)
ax.set_yticklabels(frota_por_mun.index.str.title())
ax.invert_yaxis()
ax.set_xlabel("Frota total de veículos")
ax.set_title("Frota por município do corredor")
ax.legend()

ax = axes[1]
categorias = ["Frota\n(2020→2025, 5 anos)", "Acidentes no corredor\n(2024→2025, 1 ano)"]
valores = [crescimento_frota, crescimento_acid_1ano]
cores = [COR_BASE, COR_DESTAQUE]
barras = ax.bar(categorias, valores, color=cores)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Variação (%)")
ax.set_title("Frota cresce; acidentes não acompanham no mesmo ritmo")
for b, v in zip(barras, valores):
    ax.text(b.get_x() + b.get_width() / 2, v + (1 if v >= 0 else -2), f"{v:+.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(FIGS_DIR / "fig2_frota_x_acidentes.png", dpi=150)
plt.close()

### 3 - Perfil horario de pico (pendularidade) + populacao residente 2022

In [ ]:
print("\n" + "=" * 70)
print("3 - PERFIL HORÁRIO DE PICO (PROXY DE PENDULARIDADE) + POPULAÇÃO 2022")
print("=" * 70)

hora = pd.to_datetime(corredor["horario"], format="%H:%M:%S", errors="coerce").dt.hour
hora_counts = hora.value_counts().sort_index()
pico = hora.isin([7, 8, 17, 18, 19])
pct_pico = pico.mean() * 100
print(f"Acidentes em horário de pico (7-8h, 17-19h): {pct_pico:.1f}% do total do corredor")
print("(5 das 24 horas do dia = 20,8% do tempo; {:.1f}% dos acidentes em {:.1f}% do tempo "
      "indica concentração pendular, não só lazer)".format(pct_pico, 5 / 24 * 100))

pop_path = DADOS_DIR / "populacao_corredor.json"
pop_corredor = None
if pop_path.exists():
    import json
    raw = json.load(open(pop_path, encoding="utf-8"))[1:]
    pop_corredor = pd.DataFrame(raw)
    pop_corredor["municipio"] = pop_corredor["D1N"].str.replace(r"\s*\(SC\)", "", regex=True)
    pop_corredor["populacao"] = pd.to_numeric(pop_corredor["V"], errors="coerce")
    pop_total = pop_corredor["populacao"].sum()
    print(f"População residente 2022 no corredor (18 municípios, Censo IBGE): {pop_total:,.0f} habitantes")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
cores_hora = [COR_DESTAQUE if h in [7, 8, 17, 18, 19] else COR_BASE for h in hora_counts.index]
ax.bar(hora_counts.index, hora_counts.values, color=cores_hora)
ax.set_xlabel("Hora do dia")
ax.set_ylabel("Acidentes (3 anos, corredor)")
ax.set_title(f"Picos 7-8h/17-19h concentram {pct_pico:.0f}% dos acidentes")
ax.set_xticks(range(0, 24, 2))

ax = axes[1]
if pop_corredor is not None:
    pc = pop_corredor.sort_values("populacao", ascending=False)
    ax.barh(pc["municipio"], pc["populacao"], color=COR_BASE)
    ax.invert_yaxis()
    ax.set_xlabel("População residente (Censo 2022)")
    ax.set_title(f"População do corredor: {pop_total:,.0f} hab. (Censo 2022)")
else:
    ax.text(0.5, 0.5, "Dados de população não encontrados\n(dados/populacao_corredor.json)",
            ha="center", va="center")

plt.tight_layout()
plt.savefig(FIGS_DIR / "fig3_hora_pico_populacao.png", dpi=150)
plt.close()

### 4 - Tipo de veiculo envolvido (proxy de trafego de carga) -- 2026

In [ ]:
print("\n" + "=" * 70)
print("4 - TIPO DE VEÍCULO ENVOLVIDO NO CORREDOR (proxy de tráfego de carga, 2026)")
print("=" * 70)

CAMINHAO_TIPOS = {"Caminhão", "Caminhão-trator", "Caminhão-tanque", "Reboque", "Semireboque"}
tipo_counts = corredor_ac2026["tipo_veiculo"].value_counts()
pct_caminhao = corredor_ac2026["tipo_veiculo"].isin(CAMINHAO_TIPOS).mean() * 100
print(f"Veículos tipo caminhão/reboque em acidentes do corredor (jan-mai/2026): {pct_caminhao:.1f}%")
print("Top tipos de veículo:")
print(tipo_counts.head(8).to_string())

ac2026_com_mes = corredor_ac2026.copy()
ac2026_com_mes["data_inversa"] = pd.to_datetime(ac2026_com_mes["data_inversa"], format="%Y-%m-%d", errors="coerce")
ac2026_com_mes["mes"] = ac2026_com_mes["data_inversa"].dt.month
ac2026_com_mes["e_caminhao"] = ac2026_com_mes["tipo_veiculo"].isin(CAMINHAO_TIPOS)
pct_caminhao_por_mes = ac2026_com_mes.groupby("mes")["e_caminhao"].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
top = tipo_counts.head(8)
cores_tipo = [COR_DESTAQUE if t in CAMINHAO_TIPOS else COR_BASE for t in top.index]
ax.barh(top.index[::-1], top.values[::-1], color=cores_tipo[::-1])
ax.set_xlabel("Veículos envolvidos em acidentes (corredor, jan-mai/2026)")
ax.set_title(f"{pct_caminhao:.0f}% dos veículos envolvidos são caminhão/reboque")

ax = axes[1]
meses_nomes = ["Jan", "Fev", "Mar", "Abr", "Mai"]
ax.bar(meses_nomes[:len(pct_caminhao_por_mes)], pct_caminhao_por_mes.values, color=COR_DESTAQUE, alpha=0.85)
ax.axhline(pct_caminhao, color="black", ls="--", lw=1, label=f"Média: {pct_caminhao:.1f}%")
ax.set_ylabel("% de veículos tipo caminhão/reboque")
ax.set_title("Participação de caminhões é estável mês a mês\n(não concentrada em alta temporada -- jan-mai é baixa temporada)")
ax.legend()

plt.tight_layout()
plt.savefig(FIGS_DIR / "fig4_tipo_veiculo.png", dpi=150)
plt.close()

print("\nContexto qualitativo (não auditável, sem série ANTAQ bruta -- ver dados/README.md):")
print("  Portos catarinenses: São Francisco do Sul ~4,2 Mt/ano, Itapoá ~4,0 Mt/ano, Portonave ~2,5 Mt/ano,")
print("  Imbituba ~1,7 Mt/ano, Itajaí ~0,9 Mt/ano (números de imprensa ANTAQ/SPAF-SC, não baixados como dado bruto).")

### 5 - Qualidade do pavimento (CNT 2025) -- nivel estadual

In [ ]:
print("\n" + "=" * 70)
print("5 - QUALIDADE DO PAVIMENTO (CNT 2025) -- NÍVEL ESTADUAL")
print("=" * 70)

import pypdf

reader = pypdf.PdfReader(DADOS_DIR / "cnt_pesquisa_rodovias_2025.pdf")
texto_brasil = reader.pages[0].extract_text()
texto_sc = reader.pages[26].extract_text()


def extrair_estado_geral(texto):
    otimo = re.search(r"(\d+,\d+)%[^%]*?timo", texto)
    bom = re.search(r"(\d+,\d+)%\s*Bom", texto)
    return otimo.group(1) if otimo else None, bom.group(1) if bom else None


otimo_br, bom_br = extrair_estado_geral(texto_brasil)
otimo_sc, bom_sc = extrair_estado_geral(texto_sc)
print(f"Brasil 2025: {otimo_br}% ótimo, {bom_br}% bom (ótimo+bom)")
print(f"Santa Catarina 2025: {otimo_sc}% ótimo, {bom_sc}% bom (ótimo+bom)")
print("Aviso: síntese CNT é por ESTADO, não por trecho -- não permite isolar o pavimento")
print("especificamente no corredor Garuvá-Palhoça. Usado só como teto/contexto.")

fig, ax = plt.subplots(figsize=(6, 4.5))
labels = ["Brasil", "Santa Catarina"]
otimo_vals = [float(otimo_br.replace(",", ".")), float(otimo_sc.replace(",", "."))]
bom_vals = [float(bom_br.replace(",", ".")), float(bom_sc.replace(",", "."))]
x = np.arange(len(labels))
ax.bar(x, otimo_vals, color=COR_BASE, label="Ótimo")
ax.bar(x, bom_vals, bottom=otimo_vals, color="#8fb8e8", label="Bom")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("% da extensão avaliada")
ax.set_title("Qualidade do pavimento (CNT 2025) -- nível estadual apenas")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig5_pavimento_cnt.png", dpi=150)
plt.close()

print("\n" + "=" * 70)
print("Figuras salvas em:", FIGS_DIR)
print("=" * 70)